## Identifications des éléments d'adresses 

A partir de l'adresse brute, on va essayer d'identifier les différents élements et leur position dans l'adresse. 

[NUM RUE] [TYPE DE RUE] [NOM DE RUE] [CODE POST] [VILLE] 

In [1]:
import pandas as pd
import re
 
import numpy as np


In [4]:
df = pd.read_csv("../../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv", sep=";")

df = df.dropna(subset='adresse')

df_simplified = df[['pseudo_provisoire','requete','adresse','codepost','nom_commune_postal']]


/tmp/ipykernel_10834/874753909.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv", sep=";")


On modifie le type de nos colonnes adresse et requete afin ensuite de pouvoir faire du traitement du language (TAL). 

1 - Mettre en majuscule la chaîne de caractère \
2 - Bien séparer les différents éléments (fusion de mot et caractères spéciaux)
2 - Supprimer les caractères speciaux 

In [23]:
%run geocoding_functions.py
%run cleaning_functions.py

In [24]:
df_simplified['adresse'] = df_simplified['adresse'].astype(str).apply(lambda x: x.upper())
df_simplified['requete'] = df_simplified['requete'].astype(str).apply(lambda x: x.upper())

df_simplified = prep_adresse(df_simplified)



A present nous allons chercher à repérer les tokens "usuels" dans nos adresses. Ceux ci ont été identifié et classé manuellement à partir de notre ensemble d'adresse

In [25]:
voirie = ["GRANDE RUE" ,"RUE","AVENUE", "BOULEVARD","VOIE","RUELLE","ESPLANADE",
          "PLACE","SQUARE","SQ","PL","COURS","COUR",
          "IMPASSE", "ALLEE", "CHEMIN" ,"ROUTE","RTE","IMP","PROMENADE","ALLE","ALL",
          "BD","AVE","BLD","BVD","BLV","AVN","BV",
          "FERME","DOMAINE","QUARTIER","QUR","AV","COTE","AV.","VILLA","ROND POINT", "LIEU DIT",'AVEUE','SENTIER'] # = deux mots => pas possible de trouver les deux ? 

numeros = ["1","2","3","4","5","6","7","8","9","0"]

bruit = ["APPT","BAT","CENTRE","CHEZ","HOPITAL","HOTEL","MAISON","MME","MR", "QUARTIER","RES","RESIDENCE","RETRAITE",'ETG','ETAGE','APP','BATIMENT']  #"HOP","TRANSFERT","APT","SANTE", "LOTISSEMENT"

Cas particuliers :  
=> voirie à deux mots : LIEU DIT / GRANDE RUE  
=> numéro accompagné de position 'BIS' 'TER' => supprimer ou fonction de traitement? 

In [26]:
df_simplified['voirie'] = df_simplified.apply(lambda x: find_attribute(voirie, x['adresse']), axis=1)
df_simplified['numeros'] = df_simplified.apply(lambda x: find_attribute(numeros, x['adresse'],is_number=True), axis=1)
df_simplified['bruit'] = df_simplified.apply(lambda x: find_attribute(bruit, x['adresse']), axis=1)

/tmp/ipykernel_10834/2098498829.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_simplified['voirie'] = df_simplified.apply(lambda x: find_attribute(voirie, x['adresse']), axis=1)
/tmp/ipykernel_10834/2098498829.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_simplified['numeros'] = df_simplified.apply(lambda x: find_attribute(numeros, x['adresse'],is_number=True), axis=1)
/tmp/ipykernel_10834/2098498829.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

- On ne garde que les premiers numéros apparaissant dans l'adresse en supposant que cela corresponde au numéro de rue
- On ne garde que la première voirie dans l'adresse en supposant que cela corresponde bien à la voirie.

On part du principe qu'une adresse n'a qu'un seul numéro et voirie.  

In [35]:
%run cleaning_functions.py

In [ ]:
df_simplified = filtre_num_voirie(df_simplified)

In [29]:
df_simplified

,pseudo_provisoire,requete,adresse,codepost,nom_commune_postal,voirie,numeros,bruit
0,1,34 RUE DES FRERES CHAUSSONS 92600 ASNI...,34 RUE DES FRERES CHAUSSONS,92600.0,ASNIERES-SUR-SEINE,RUE,34,
1,2,11 RUE EMILE DUBOIS 75014 PARIS,11 RUE EMILE DUBOIS,75014.0,PARIS,RUE,11,
2,3,48 CHEMIN VERT 78680 EPONE,48 CHEMIN VERT,78680.0,EPONE,CHEMIN,48,
3,4,18 ALLEE DE LA CHARNILLE 47140 SAIN...,18 ALLEE DE LA CHARNILLE,47140.0,SAINT-SYLVESTRE-SUR-LOT,ALLEE,18,
4,5,31 RUE DU GENERAL DE MIRIBEL 92500 RUEI...,31 RUE DU GENERAL DE MIRIBEL,92500.0,RUEIL-MALMAISON,RUE,31,
...,...,...,...,...,...,...,...,...
57873,64290,3 AV DE FOUILLEUSE 92210 SAIN...,3 AV DE FOUILLEUSE,92210,SAINT-CLOUD,AV,3,
57874,64291,81 COTE DU TORCHON 27220 L'HABIT,81 COTE DU TORCHON,27220,L'HABIT,COTE,81,
57875,64292,60 RUE BAUDRICOURT 75013 PARI...,60 RUE BAUDRICOURT,75013,Paris 13,RUE,60,
57876,64293,159 AVENUE DE LA REPUBLIQUE 92320 CHAT...,159 AVENUE DE LA REPUBLIQUE,92320,CHATILLON,AVENUE,159,


Une fois les éléments trouvés on veut maintenant obtenir leur position dans l'adresse :

PS : Le travail étant initialement réaliser pour Doccano il faut compter un décalage de 1 entre la position de python et la position réelle (pas de position 0)

In [ ]:
df_simplified['pos_voirie'] = df_simplified.apply(lambda x: find_pos_attributes(x['voirie'], x['adresse']), axis=1)
df_simplified['pos_numeros'] = df_simplified.apply(lambda x: find_pos_attributes(x['numeros'], x['adresse'],is_number=True), axis=1)
df_simplified['pos_bruit'] = df_simplified.apply(lambda x: find_pos_attributes(x['bruit'], x['adresse']), axis=1)
            

In [ ]:
# def filtre_multiple_pos_voirie(df):

#     for i in df[df['voirie']!=""].index:

#         pos_voirie = df.loc[i,'pos_voirie'].split(',')
#         if len(voirie) >1 :
#             df.loc[i,'pos_voirie'] = pos_voirie[0]
#     return df 


# df_simplified = filtre_multiple_pos_voirie(df_simplified)

On a remarqué que certains éléments ayant été attrapé comme du bruit sont en réalité une voirie (ex: résidence).  
On suppose que si le bruit est précédé d'un numéro, alors celui ci est un nom de voirie

normalement tous les éléments taggés ont leur position associée

In [31]:
df_simplified = filtre_bruit_to_voirie(df_simplified) 


Fonction permettant de regarder l'ensemble des adresses contenant un mot donné 

En raison de l'ambiguité de la langue française certains éléments n'ont pas été considérés comme des voiries mais le sont dans certains cas.  
Ex : VILLA  
On est parti de l'hypothèse que si l'élément était précédé d'un numéro alors on pouvait le considérer comme une voirie   

In [32]:
df_simplified = clean_wrong_voirie(df_simplified)


#### On cherche les nom de rue : 
- revient au travail de clean up pour l'ensemble de nos données 

D'abord on nettoie le bruit des adresses pour ensuite filtrer les noms de voirie 

In [ ]:
df_simplified['pos_prc_voirie'] = df_simplified.apply(lambda x: find_pos_prc_attributes(x['voirie'], x['adresse']), axis=1)
df_simplified['pos_prc_numeros'] = df_simplified.apply(lambda x: find_pos_prc_attributes(x['numeros'], x['adresse'],is_number=True), axis=1)
df_simplified['pos_prc_bruit'] = df_simplified.apply(lambda x: find_pos_prc_attributes(x['bruit'], x['adresse']), axis=1)       

In [34]:
df = filtre_elem_adresse(df_simplified)

ValueError: invalid literal for int() with base 10: '2,8'

## FIN ALGO ID ADRESSE 

In [ ]:
# def word_in_adresse(df,mot):
#     return df[df['adresse'].str.contains(mot,case=False,na=False)]


In [ ]:
df.to_csv("../data/data_cleaned/biais_cleaned/patients_adresse_id.csv", sep=";")
